In [1]:
%pip install datasets math_verify vllm torch sympy

Note: you may need to restart the kernel to use updated packages.


In [2]:
from datasets import load_dataset

ds = load_dataset("open-r1/OpenR1-Math-220k", "default")
ds

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['problem', 'solution', 'answer', 'problem_type', 'question_type', 'source', 'uuid', 'is_reasoning_complete', 'generations', 'correctness_math_verify', 'correctness_llama', 'finish_reasons', 'correctness_count', 'messages'],
        num_rows: 93733
    })
})

In [3]:
import gc
import torch
from vllm import LLM, SamplingParams
from math_verify import parse, verify
import sympy

for var in ["llm", "generations"]:
    if var in globals():
        del globals()[var]
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()


llm = LLM(
    model="deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
    gpu_memory_utilization=0.6,
)
sampling_params = SamplingParams(
    n=8,
    max_tokens=2**13,
)

preview_ds = ds["train"][1:2]
prompts = preview_ds["problem"]
golds = preview_ds["answer"]

generations = llm.generate(prompts, sampling_params)
for prompt, gold, generation in zip(prompts, golds, generations):
    print("*"*50 + " Prompt " + "*"*50)
    print(prompt)
    
    for i, output in enumerate(generation.outputs):
        text = output.text
        answer = parse(text)
        correct = verify(parse(gold), answer)
        
        print("*"*50 + f" Generation {i+1}: {sympy.latex(answer)} ({"correct" if correct else "incorrect"}) ", "*"*50)
        if correct:
            print(text)

INFO 08-23 22:01:36 [api_utils.py:273] non-default args: {'gpu_memory_utilization': 0.6, 'disable_log_stats': True, 'model': 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B'}
INFO 08-23 22:01:36 [model.py:645] Resolved architecture: Qwen2ForCausalLM
WARNING 08-23 22:01:36 [model.py:2164] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 08-23 22:01:36 [model.py:2217] Casting torch.bfloat16 to torch.float16.
INFO 08-23 22:01:36 [model.py:1883] Using max model len 131072
INFO 08-23 22:01:36 [scheduler.py:242] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 08-23 22:01:36 [kernel.py:306] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
WARNING 08-23 22:01:40 [system_utils.py:157] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en

[W823 22:01:55.263637656 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


(EngineCore pid=2066) INFO 08-23 22:01:57 [model_runner.py:308] Loading model from scratch...
(EngineCore pid=2066) ERROR 08-23 22:01:57 [fa_utils.py:273] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
(EngineCore pid=2066) INFO 08-23 22:02:01 [cuda.py:482] Using TRITON_ATTN attention backend out of potential backends: ['TRITON_ATTN', 'FLEX_ATTENTION'].
(EngineCore pid=2066) INFO 08-23 22:02:02 [weight_utils.py:867] Filesystem type for checkpoints: OVERLAY. Checkpoint size: 3.31 GiB. Available RAM: 11.04 GiB.
(EngineCore pid=2066) INFO 08-23 22:02:02 [weight_utils.py:890] Auto-prefetch is disabled because the filesystem (OVERLAY) is not a recognized network FS (NFS/Lustre). If you want to force prefetching, start vLLM with --safetensors-load-strategy=prefetch.


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:02<00:00,  2.42s/it]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:02<00:00,  2.42s/it]
(EngineCore pid=2066) 


(EngineCore pid=2066) INFO 08-23 22:02:05 [default_loader.py:430] Loading weights took 2.54 seconds
(EngineCore pid=2066) INFO 08-23 22:02:06 [model_runner.py:329] Model loading took 3.45 GiB and 9.308450 seconds
(EngineCore pid=2066) WARNING 08-23 22:02:06 [topk_topp_sampler.py:69] FlashInfer top-p/top-k sampling unavailable: unsupported compute capability 7.5; falling back. Set VLLM_USE_FLASHINFER_SAMPLER=0 to silence.
(EngineCore pid=2066) INFO 08-23 22:02:08 [caching.py:335] reconstructed serializable fn from standalone compile artifacts. num_artifacts=3 num_submods=29
(EngineCore pid=2066) INFO 08-23 22:02:08 [decorators.py:311] Directly load AOT compilation from path /root/.cache/vllm/torch_compile_cache/torch_aot_compile/af68ece8784a1aa24e6a420cecb0d4c710dcfcb24120464d45f2e760c657287a/rank_0_0/model
(EngineCore pid=2066) INFO 08-23 22:02:08 [monitor.py:53] torch.compile took 0.21 s in total
(EngineCore pid=2066) INFO 08-23 22:02:08 [monitor.py:81] Initial profiling/warmup run to

...

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/8 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

(EngineCore pid=2066) WARNING 08-23 22:02:22 [jit_monitor.py:135] Triton kernel JIT compilation during inference: kernel_unified_attention. This causes a latency spike; consider extending warmup to cover this shape/config.


Processed prompts: 100%|██████████| 8/8 [04:20<00:00, 32.61s/it, est. speed input: 3.43 toks/s, output: 192.99 toks/s]


************************************************** Prompt **************************************************
3. (6 points) A construction company was building a tunnel. When $\frac{1}{3}$ of the tunnel was completed at the original speed, they started using new equipment, which increased the construction speed by $20 \%$ and reduced the working hours to $80 \%$ of the original. As a result, it took a total of 185 days to complete the tunnel. If they had not used the new equipment and continued at the original speed, it would have taken $\qquad$ days to complete the tunnel.
************************************************** Generation 1: \left[ 0.0033, \  \mathtt{\text{0.0033}}\right] (incorrect)  **************************************************
************************************************** Generation 2: \left[ 5, \  \mathtt{\text{5}}\right] (incorrect)  **************************************************
************************************************** Generation 3: \left[ 208.